# Artículo: Limpieza de datos de grupos criminales (OCVED)
* **Nombre**: Sierra Wells  
* **Clase**: Programación (Geointeligencia Computacional)
* **Programa**: Maestría en Ciencias de Información Geoespacial
* **Fecha**: 19 noviembre 2026

## 📝 Descripción
Lee la base Organized Criminal Violence Event Data (OCVED) 2.0 y construye, para cada **municipio**, predictores sobre la presencia violenta de grupos criminales en 2017, un año antes de los homicidios que se modelan (2018):
* `n_criminal_groups`: número de grupos criminales identificados distintos en el municipio.
* Variables _dummy_ (`group_...`) para los 10 grupos criminales identificados con más registros en 2017: 1 si el grupo aparece en el municipio, 0 si no.

Los municipios son los del censo 2010: los registros en municipios creados después de 2010 se asignan a su municipio padre, según `clean/hand/mun_crosswalk.yaml`.

Fuente: [OCVED 2.0](https://github.com/javierosorio/OCVED_2.0) (Osorio y Beltrán, 2020).

Resultado en `clean/output/`:
* `3_clean_ocved.csv`: predictores por municipio (`cve_mun`), solo de los municipios con al menos un grupo identificado.

In [1]:
# Librerías
from pathlib import Path

import pandas as pd
import yaml

In [2]:
# Rutas (el notebook se corre desde clean/src/)
TASK_DIR = Path.cwd().parent
INPUT_DIR = TASK_DIR / 'input'
HAND_DIR = TASK_DIR / 'hand'
OUTPUT_DIR = TASK_DIR / 'output'

In [3]:
# Año de OCVED que se usa (un año antes de los homicidios de 2018)
YEAR = 2017

# Número de grupos criminales con variable dicotómica
N_TOP_GROUPS = 10

# Etiquetas genéricas de actor_sub que no identifican a un grupo criminal específico:
# todos los subgrupos de "Unidentified Criminal Group", "Individual" (de "Other") y
# "Huachicoleros" (describe la actividad de robo de combustible, no a un grupo)
GENERIC_LABELS = [
    'Unidentified Criminal Group', 'Criminal', 'Hitman', 'Cartel member', 'Leader',
    'Drugtrafficker', 'Key operator', 'Individual', 'Huachicoleros',
]

# Subgrupos que son nombres de líderes o familias de una organización: se cuentan como la organización
LEADER_LABELS = {
    'El Chapo': 'Cartel de Sinaloa',
    'Zambada': 'Cartel de Sinaloa',
    'Nacho Coronel': 'Cartel de Sinaloa',
    'El Guero Palma': 'Cartel de Sinaloa',
    'Arellano Felix': 'Cartel de Tijuana',
}

**1. Leer datos**

Según la metodología del OCVED 2.0, cada fila es un grupo criminal (`actor_sub`, subgrupo de la organización principal `actor_main`) detectado en un municipio en un día, en noticias sobre violencia del crimen organizado. Los autores eliminan duplicados, así que cada subgrupo aparece una sola vez por municipio y día.

In [4]:
ocved = pd.read_excel(INPUT_DIR / 'OCVED_2.0.xlsx', sheet_name='Criminals_v2 0')

# Revisión: no hay subgrupos repetidos en el mismo municipio y día
assert not ocved.duplicated(['date', 'mun', 'actor_sub']).any()

ocved.head()

,date,date_elapsed,year,month,day,state,mun,counter,actor_main,actor_sub,longitude,latitude
0,20000105,2000-01-05,2000,1,5,7,7024,1,Other,Los Campesinos,-92.257500,16.6575
1,20000105,2000-01-05,2000,1,5,11,11020,1,Other,Los Campesinos,-101.680556,21.119722
2,20000105,2000-01-05,2000,1,5,7,7059,1,Other,Los Campesinos,-92.096111,16.907222
3,20000106,2000-01-06,2000,1,6,28,28027,1,Unidentified Criminal Group,Unidentified Criminal Group,-99.508056,27.486389
4,20000112,2000-01-12,2000,1,12,7,7004,1,Other,Individual,-92.038333,16.735556


**2. Quedarse con 2017**

In [5]:
ocved_year = ocved.loc[ocved['year'] == YEAR].copy()

# Claves de entidad (2 dígitos) y de municipio (5 dígitos); en OCVED son números, por ejemplo 7 y 7001
ocved_year['cve_ent'] = ocved_year['state'].astype(str).str.zfill(2)
ocved_year['cve_mun'] = ocved_year['mun'].astype(str).str.zfill(5)

# Revisión: la entidad coincide con los primeros dígitos de la clave de municipio
assert (ocved_year['cve_mun'].str[:2] == ocved_year['cve_ent']).all()

**3. Quitar etiquetas genéricas**

Las etiquetas genéricas (por ejemplo, "Hitman" o "Individual") no identifican a un grupo criminal específico, así que se excluyen. Se usa `actor_sub` para identificar a los grupos: si `actor_main` es "Other" y `actor_sub` es "Los Limones", Los Limones cuenta como un grupo identificado.

In [6]:
identified = ocved_year.loc[~ocved_year['actor_sub'].isin(GENERIC_LABELS)].copy()

Algunos subgrupos son nombres de líderes o familias de una organización (por ejemplo, `El Chapo` en el Cártel de Sinaloa). Se reemplazan por el nombre de su organización para no contarlos como grupos distintos. Si después del reemplazo un grupo aparece dos veces en el mismo municipio y día, se deja una sola fila, igual que en la limpieza de OCVED.

In [7]:
identified['actor_sub'] = identified['actor_sub'].replace(LEADER_LABELS)
identified = identified.drop_duplicates(['date', 'mun', 'actor_sub'])

identified[['cve_ent', 'cve_mun', 'actor_main', 'actor_sub']].head()

,cve_ent,cve_mun,actor_main,actor_sub
51464,12,12074,Beltran Leyva,Beltran Leyva
51466,16,16006,La Familia Michoacana,Los Viagras
51468,12,12074,La Barbie,Los Ardillos
51472,12,12042,Beltran Leyva,Beltran Leyva
51473,12,12042,La Barbie,Los Ardillos


**4. Construir predictores por municipio**

Las variables dicotómicas son para los 10 grupos identificados con más registros en todo el país.

In [8]:
# Los 10 grupos identificados con más filas en 2017, y sus nombres de columna (por ejemplo, group_los_zetas)
top_groups = identified['actor_sub'].value_counts().head(N_TOP_GROUPS).index
group_columns = {group: 'group_' + group.lower().replace(' ', '_') for group in top_groups}

list(group_columns.values())

['group_los_zetas',
 'group_cartel_de_sinaloa',
 'group_cartel_de_jalisco_nueva_generacion',
 'group_la_familia_michoacana',
 'group_beltran_leyva',
 'group_cartel_del_golfo',
 'group_caballeros_templarios',
 'group_la_linea',
 'group_cartel_de_tijuana',
 'group_cartel_de_tlahuac']

In [9]:
# Municipios creados después de 2010 y su municipio padre (clean/hand/mun_crosswalk.yaml)
with open(HAND_DIR / 'mun_crosswalk.yaml', encoding='utf-8') as file:
    new_muns = pd.DataFrame(yaml.safe_load(file)['new_municipalities'])

# Reemplazamos la clave de cada municipio nuevo por la de su padre
parent_of = dict(zip(new_muns['cve_mun_new'], new_muns['cve_mun_2010']))
identified['cve_mun'] = identified['cve_mun'].replace(parent_of)

In [10]:
# Número de grupos identificados distintos y variable dicotómica por grupo, por municipio
# (solo los municipios con al menos un grupo identificado; los ceros se agregan en replicate)
n_groups_mun = identified.groupby('cve_mun')['actor_sub'].nunique().rename('n_criminal_groups')
dummies_mun = pd.crosstab(identified['cve_mun'], identified['actor_sub'])[top_groups].gt(0).astype(int)

ocved_mun = (
    pd.concat([n_groups_mun, dummies_mun], axis=1)
    .rename(columns=group_columns)
    .rename_axis('cve_mun')
    .reset_index()
)

ocved_mun.head()

,cve_mun,n_criminal_groups,group_los_zetas,group_cartel_de_sinaloa,group_cartel_de_jalisco_nueva_generacion,group_la_familia_michoacana,group_beltran_leyva,group_cartel_del_golfo,group_caballeros_templarios,group_la_linea,group_cartel_de_tijuana,group_cartel_de_tlahuac
0,01002,1,0,0,0,0,0,0,0,0,0,0
1,02001,2,0,1,1,0,0,0,0,0,0,0
2,02002,6,0,1,1,0,1,0,0,0,1,0
3,02003,2,0,1,1,0,0,0,0,0,0,0
4,02004,6,0,1,1,1,0,0,1,0,1,0


**5. Guardar resultados**

In [11]:
ocved_mun.to_csv(OUTPUT_DIR / '3_clean_ocved.csv', index=False)